# ResNet — residual connections beat the degradation problem

> Tutorial pair for [`resnet.py`](resnet.py).

## 1. Intuition
Stacking more layers *should* never hurt: the extra layers could just learn the
identity. In practice, plain very-deep CNNs do worse — training error goes **up**
past a certain depth. This is the **degradation problem**, and it is an
optimization failure (gradients struggle to reach early layers), not overfitting.
ResNet's fix is tiny: have each block learn a **residual** $F(x)=H(x)-x$ and emit
$x+F(x)$. The identity shortcut hands every layer a direct gradient highway, so
100+ layer nets train fine.

## 2. Concept (the slide)
- **Residual block:** `out = x + F(x)`, where $F$ is a couple of conv-BN-ReLU
  layers. Learning a *perturbation* of the identity is easier than learning the
  whole mapping from scratch.
- **BasicBlock** (ResNet-18/34): two $3\times3$ convs + skip.
- **Bottleneck** (ResNet-50/101/152): $1\times1 \to 3\times3 \to 1\times1$, so the
  expensive $3\times3$ runs in a *reduced* channel space (cheap), then channels
  are restored.
- **Projection shortcut:** when a block changes spatial size or channel count, the
  skip uses a $1\times1$ conv so the shapes match before adding.
- **BatchNorm** keeps activations well-scaled (see `06.training-techniques/README.md`).

## 3. Math — why the residual gradient never vanishes

**Forward.** A residual block computes
$$y = x + F(x;\,\mathcal W).$$

**Backward (the whole point).** Differentiate w.r.t. the input:
$$\frac{\partial y}{\partial x} = I + \frac{\partial F}{\partial x}
   = I + F'(x).$$
The $+I$ is an **identity gradient path**. Now chain $L$ blocks; with upstream
loss $\mathcal L$ and intermediate activations $x_l$,
$$\frac{\partial \mathcal L}{\partial x_0}
  = \frac{\partial \mathcal L}{\partial x_L}\prod_{l=1}^{L}\big(I + F_l'(x_{l-1})\big).$$
Expanding the product, **one term is the bare identity** $\prod I = I$: the
gradient at the deepest layer reaches the input *undiminished*, plus correction
terms. Contrast the **plain** net, where
$$\frac{\partial \mathcal L}{\partial x_0}
  = \frac{\partial \mathcal L}{\partial x_L}\prod_{l=1}^{L} F_l'(x_{l-1}),$$
a pure product of Jacobians whose magnitude scales **geometrically** — if each
$\lVert F_l'\rVert<1$ it vanishes, if $>1$ it explodes. The $+I$ shortcut converts
that multiplicative chain into an additive one, which is the cure.

**Bottleneck parameter math.** A $3\times3$ conv with $C$ in/out channels costs
$9C^2$ weights. The bottleneck wraps it as $1\times1$ (reduce $C\to C/4$),
$3\times3$ (in $C/4$), $1\times1$ (restore $C/4\to C$):
$$C\cdot\tfrac{C}{4} + 9\cdot\big(\tfrac{C}{4}\big)^2 + \tfrac{C}{4}\cdot C
  = \tfrac{C^2}{4}+\tfrac{9C^2}{16}+\tfrac{C^2}{4}
  = \tfrac{17}{16}C^2 \approx 1.06\,C^2,$$
versus a *two* $3\times3$ BasicBlock at $18C^2$ — far cheaper for the same depth.

## 4. Key building block — BasicBlock & Bottleneck

In [ ]:
# ===== actual implementation from resnet.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def residual_gradient_demo(n_layers: int = 50, dim: int = 16, seed: int = SEED):
    r"""Measure how the input-gradient magnitude survives depth, with vs without
    a skip connection — using a tiny hand-written backward pass (no autograd).

    Each "layer" is z = tanh(x @ W); a residual layer outputs x + z. We backprop
    a unit upstream gradient through `n_layers` and report ||dL/dx_input||.

    Residual identity (the whole point):
        out = x + F(x)  =>  d(out)/d(x) = I + F'(x).
    The "+ I" guarantees a gradient highway of magnitude ~1 regardless of depth.
    """
    rng = np.random.default_rng(seed)
    # Small weights so the plain net's Jacobian product shrinks (vanishing).
    Ws = [rng.normal(0, 0.5, (dim, dim)) for _ in range(n_layers)]

    def run(x0, residual: bool):
        # forward, caching pre-activations
        xs, zs = [x0], []
        x = x0
        for W in Ws:
            z = x @ W
            zs.append(z)
            a = np.tanh(z)
            x = x + a if residual else a
            xs.append(x)
        # backward: start from a unit upstream gradient on the output
        g = np.ones_like(x)
        norms = [np.linalg.norm(g)]
        for l in reversed(range(n_layers)):
            dz = g * (1.0 - np.tanh(zs[l]) ** 2)     # through tanh
            dx_through_F = dz @ Ws[l].T              # through the weight
            g = dx_through_F + g if residual else dx_through_F  # "+ I" skip path
            norms.append(np.linalg.norm(g))
        norms.reverse()                              # index 0 = input layer
        return np.array(norms)

    x0 = rng.normal(0, 1, (4, dim))
    plain = run(x0, residual=False)
    resid = run(x0, residual=True)
    return plain, resid

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def conv3x3(in_c: int, out_c: int, stride: int = 1) -> nn.Conv2d:
    return nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)

def conv1x1(in_c: int, out_c: int, stride: int = 1) -> nn.Conv2d:
    return nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False)

class BasicBlock(nn.Module):
    """Two stacked 3x3 convs with a skip: out = ReLU( x_proj + F(x) ).

    F(x) = BN(conv3x3( ReLU( BN(conv3x3(x)) ) )). When the block changes spatial
    size (stride>1) or channel count, the shortcut uses a 1x1 conv to match.
    """

    expansion = 1

    def __init__(self, in_c: int, out_c: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_c, out_c, stride)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = conv3x3(out_c, out_c)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut: nn.Module = nn.Identity()
        if stride != 1 or in_c != out_c * self.expansion:
            self.shortcut = nn.Sequential(
                conv1x1(in_c, out_c * self.expansion, stride),
                nn.BatchNorm2d(out_c * self.expansion),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)          # the residual addition (the +x path)
        return F.relu(out)

class Bottleneck(nn.Module):
    """1x1 -> 3x3 -> 1x1 with a skip (ResNet-50+). The 1x1 convs squeeze then
    restore channels, so the expensive 3x3 runs in a low-dimensional space —
    far fewer FLOPs/params for the same representational power."""

    expansion = 4

    def __init__(self, in_c: int, mid_c: int, stride: int = 1):
        super().__init__()
        out_c = mid_c * self.expansion
        self.conv1 = conv1x1(in_c, mid_c)             # reduce
        self.bn1 = nn.BatchNorm2d(mid_c)
        self.conv2 = conv3x3(mid_c, mid_c, stride)    # spatial mixing (cheap)
        self.bn2 = nn.BatchNorm2d(mid_c)
        self.conv3 = conv1x1(mid_c, out_c)            # restore
        self.bn3 = nn.BatchNorm2d(out_c)
        self.shortcut: nn.Module = nn.Identity()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                conv1x1(in_c, out_c, stride), nn.BatchNorm2d(out_c))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = out + self.shortcut(x)
        return F.relu(out)

## 5. Full architecture (PyTorch) — a small ResNet from stages of blocks

In [ ]:
# ===== actual implementation from resnet.py =====
class TinyResNet(nn.Module):
    """A small ResNet for tiny images. `block` is BasicBlock or Bottleneck;
    `layers` gives the number of blocks per stage."""

    def __init__(self, block=BasicBlock, layers=(2, 2), in_c=1,
                 base=8, n_classes=10):
        super().__init__()
        self.in_c = base
        self.stem = nn.Sequential(
            nn.Conv2d(in_c, base, 3, padding=1, bias=False),
            nn.BatchNorm2d(base), nn.ReLU(inplace=True))
        self.stage1 = self._make_stage(block, base, layers[0], stride=1)
        self.stage2 = self._make_stage(block, base * 2, layers[1], stride=2)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(base * 2 * block.expansion, n_classes)

    def _make_stage(self, block, mid_c, n_blocks, stride):
        strides = [stride] + [1] * (n_blocks - 1)
        blocks = []
        for s in strides:
            blocks.append(block(self.in_c, mid_c, s))
            self.in_c = mid_c * block.expansion
        return nn.Sequential(*blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

class _PlainConv(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))

class PlainNet(nn.Module):
    """Same conv depth as a BasicBlock TinyResNet but WITHOUT the +x shortcuts.
    Used to expose vanishing gradients in the early layers."""

    def __init__(self, depth=16, in_c=1, width=8, n_classes=10):
        super().__init__()
        layers = [_PlainConv(in_c, width)]
        for _ in range(depth - 1):
            layers.append(_PlainConv(width, width))
        self.features = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(width, n_classes)

    def forward(self, x):
        x = self.features(x)
        return self.fc(self.pool(x).flatten(1))

class ResidualNet(nn.Module):
    """Twin of PlainNet: same conv layers, but every pair is wrapped in a skip."""

    class _ResPair(nn.Module):
        def __init__(self, width):
            super().__init__()
            self.c1 = _PlainConv(width, width)
            self.c2 = nn.Sequential(
                nn.Conv2d(width, width, 3, padding=1, bias=False),
                nn.BatchNorm2d(width))

        def forward(self, x):
            return F.relu(x + self.c2(self.c1(x)))   # identity shortcut

    def __init__(self, depth=16, in_c=1, width=8, n_classes=10):
        super().__init__()
        self.stem = _PlainConv(in_c, width)
        self.blocks = nn.Sequential(*[self._ResPair(width)
                                      for _ in range((depth - 1) // 2)])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(width, n_classes)

    def forward(self, x):
        x = self.blocks(self.stem(x))
        return self.fc(self.pool(x).flatten(1))

def layerwise_grad_norms(model: nn.Module, x: torch.Tensor,
                         y: torch.Tensor) -> list[float]:
    """Backprop one batch and return the gradient norm of each conv weight,
    ordered from input layer to output layer."""
    model.zero_grad()
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward()
    norms = []
    for m in model.modules():
        if isinstance(m, nn.Conv2d) and m.weight.grad is not None:
            norms.append(m.weight.grad.norm().item())
    return norms

def demo():
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.set_num_threads(1)        # tiny ops: avoid thread-thrashing on big CPUs
    dev = get_device()

    # --- (a) NumPy: the residual gradient identity I + F'(x) ----------------
    plain_g, resid_g = residual_gradient_demo(n_layers=50, dim=16)
    print("NumPy 50-layer tanh net — ||grad w.r.t. activations|| (input ... output):")
    print(f"  plain  : input={plain_g[0]:.2e}  output={plain_g[-1]:.2e}"
          f"  -> shrinks {plain_g[-1] / (plain_g[0] + 1e-30):.1e}x toward input")
    print(f"  residual: input={resid_g[0]:.2e}  output={resid_g[-1]:.2e}"
          f"  -> stays O(1) thanks to the + I path")

    # --- (b) PyTorch: deep PLAIN vs RESIDUAL per-layer conv grad norms -------
    x = torch.randn(8, 1, 8, 8, device=dev)
    yb = torch.randint(0, 10, (8,), device=dev)
    plain = PlainNet(depth=16, width=8).to(dev)
    resnet_twin = ResidualNet(depth=16, width=8).to(dev)
    pn = layerwise_grad_norms(plain, x, yb)
    rn = layerwise_grad_norms(resnet_twin, x, yb)
    print("\n16-conv PyTorch nets — first-layer / last-layer conv ||dW||:")
    print(f"  plain   : first={pn[0]:.2e}  last={pn[-1]:.2e}"
          f"  ratio last/first = {pn[-1] / (pn[0] + 1e-30):.1f}x")
    print(f"  residual: first={rn[0]:.2e}  last={rn[-1]:.2e}"
          f"  ratio last/first = {rn[-1] / (rn[0] + 1e-30):.1f}x")
    print("  -> skips keep the early-layer gradient from collapsing.")

    # --- (c) Build the blocks & a small ResNet, count params, train a few steps
    for name, net in [("BasicBlock ResNet", TinyResNet(BasicBlock, (2, 2))),
                      ("Bottleneck ResNet", TinyResNet(Bottleneck, (2, 2)))]:
        net = net.to(dev)
        out = net(x)
        n_params = sum(p.numel() for p in net.parameters())
        print(f"\n{name}: output {tuple(out.shape)}, params = {n_params:,}")
        opt = torch.optim.Adam(net.parameters(), lr=1e-2)
        loss_fn = nn.CrossEntropyLoss()
        losses = []
        for _ in range(15):
            opt.zero_grad()
            loss = loss_fn(net(x), yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        print(f"  loss: {losses[0]:.3f} -> {losses[-1]:.3f} (going down)")

## 6. Run — measure the gradient highway

The demo (a) backprops a unit gradient through a 50-layer tanh net in pure NumPy
with vs without skips, then (b) compares per-conv-layer gradient norms in a
16-conv PyTorch plain net vs its residual twin, then (c) builds the Basic and
Bottleneck ResNets and trains a few steps.

In [ ]:
demo()

## 7. Visualization — per-layer gradient norm: plain vs residual

The money plot: in the deep plain net the gradient **collapses** toward the input
layer; the residual twin keeps it $O(1)$ all the way down, exactly as $I+F'$
predicts.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import torch
import resnet as M

torch.manual_seed(0); torch.set_num_threads(1)

# (a) NumPy 50-layer net: activation-gradient norm vs depth
plain_g, resid_g = M.residual_gradient_demo(n_layers=50, dim=16)

# (b) PyTorch 16-conv nets: per-conv-weight grad norm, input->output
x = torch.randn(8, 1, 8, 8); y = torch.randint(0, 10, (8,))
pn = M.layerwise_grad_norms(M.PlainNet(depth=16, width=8), x, y)
rn = M.layerwise_grad_norms(M.ResidualNet(depth=16, width=8), x, y)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(plain_g, "s-", label="plain")
ax[0].semilogy(resid_g, "o-", label="residual (+I)")
ax[0].set_title("NumPy 50-layer tanh net"); ax[0].set_xlabel("layer (0=input)")
ax[0].set_ylabel(r"$\||\partial L/\partial x\||$ (log)"); ax[0].legend(); ax[0].grid(True, alpha=.3)

ax[1].semilogy(range(1, len(pn)+1), pn, "s-", label="plain")
ax[1].semilogy(range(1, len(rn)+1), rn, "o-", label="residual")
ax[1].set_title("PyTorch 16-conv nets"); ax[1].set_xlabel("conv layer (1=input)")
ax[1].set_ylabel(r"$\||dW\||$ (log)"); ax[1].legend(); ax[1].grid(True, alpha=.3)
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **Residual = additive, not multiplicative.** $\partial y/\partial x = I + F'$
  gives a guaranteed gradient path; that is *the* fix for the degradation problem.
- **Bottlenecks** make depth affordable: do the expensive $3\times3$ in a squeezed
  channel space.
- **Pitfall — shape mismatch:** if a block changes stride or channels you must
  project the shortcut (a $1\times1$ conv), or the addition will not align.
- **Pitfall — pre/post-activation:** the original block applies ReLU *after* the
  add; "Identity Mappings" (2016) moves BN/ReLU *before* the convs for an even
  cleaner identity path. Both appear in the literature.
- Related fixes for vanishing gradients live in `06.training-techniques/README.md`
  (init, BatchNorm, gated RNN memory).